In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG   = "sample_project_playground_003"
RAW_TABLE = f"{CATALOG}.raw.ecb_raw_json"
CKPT_ROOT = f"/Volumes/{CATALOG}/raw/checkpoints/ecb_raw_json"
STORAGE   = "abfss://landing@stsampleplayground003.dfs.core.windows.net"

SOURCES = {
    "ecb-historical": f"{STORAGE}/ecb_historical/",
    "ecb-daily":      f"{STORAGE}/ecb_daily/",
}

# --- DDL (idempotent) -------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RAW_TABLE} (
  payload                VARIANT,
  source_folder          STRING,
  file_path              STRING,
  file_modification_time TIMESTAMP,
  ingested_at            TIMESTAMP
)
USING DELTA
""")

# --- Auto Loader: one stream (and one checkpoint) per source folder ---------
for name, path in SOURCES.items():
    stream_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")               # one row per file
        .option("singleVariantColumn", "payload")  # whole file as VARIANT
        .load(path)
    )
    query = (
        stream_df.select(
            F.col("payload"),
            F.lit(name).alias("source_folder"),
            F.col("_metadata.file_path").alias("file_path"),
            F.col("_metadata.file_modification_time").alias("file_modification_time"),
            F.current_timestamp().alias("ingested_at"),
        )
        .writeStream
        .option("checkpointLocation", f"{CKPT_ROOT}/{name}")
        .trigger(availableNow=True)
        .toTable(RAW_TABLE)
    )
    query.awaitTermination()

# --- Raw DQ (no f-string: literal '{' is safe) ------------------------------
bad = spark.sql(
    "SELECT count(*) AS n FROM " + RAW_TABLE +
    " WHERE payload IS NULL"
    "    OR LEFT(CAST(payload AS STRING), 1) <> '{'"
)
n_bad, n_files = bad.collect()[0]["n"], spark.table(RAW_TABLE).count()
assert n_bad == 0, f"{n_bad} unparseable payload(s)"
print(f"raw OK: {n_files} files ingested, 0 bad")